In [4]:
import pandas as pd

"""
Extract data from raw csv dataset
"""

PATH = "../data/raw/news/cnbc_articles_checkpoint (2).csv"

df = pd.read_csv(PATH)
print(df.loc[3])

url                    https://news.google.com/rss/articles/CBMijAFBV...
title                                                                NaN
published_at                                                         NaN
author                                                               NaN
body                                                                 NaN
status                                                           success
error                                                                NaN
discovery_title        Op-ed: Will China's President Xi’s big bet pay...
discovery_published                        Sun, 19 Sep 2021 07:00:00 GMT
discovery_query                                site:cnbc.com geopolitics
Name: 3, dtype: object


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import gnewsdecoder

"""
Scrap html from url
"""

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; research scraper/1.0)"
}

def resolve_article_url(url):
    try:
        result = gnewsdecoder(
            url,
            interval=1,
            timeout=15
        )
        
        return result["decoded_url"]
    except Exception:
        pass

    return url

def scrape_html(url):
    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        return response.text, response.status_code, None
    except requests.RequestException as exc:
        return None, None, str(exc)

results = [None] * len(df)

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(scrape_html, resolve_article_url(url)): index
        for index, url in df["url"].items()
    }

    for future in as_completed(futures):
        results[futures[future]] = future.result()

df[["html", "scrape_http_status", "scrape_error"]] = pd.DataFrame(
    results,
    index=df.index,
    columns=["html", "scrape_http_status", "scrape_error"]
)

In [6]:
import cleaning
from bs4 import BeautifulSoup

"""
Clean the dataset, fill the title, body, and published_at
"""

def extract_and_clean(html):
    if not isinstance(html, str) or not html.strip():
        return None, pd.NaT

    soup = BeautifulSoup(html, "html.parser")

    # extract publish date
    time_tag = soup.find("time", attrs={"datetime": True})
    published_at = (
        pd.to_datetime(time_tag["datetime"], errors="coerce")
        if time_tag
        else pd.NaT
    )
    
    # extract author
    author_tag = soup.find("a", attrs={"class": "Author-authorName"})
    author = author_tag.getText(" ", strip=True) if author_tag else None
    
    # extract title
    title_tag = soup.find("title")
    title = title_tag.get_text(" ", strip=True) if title_tag else None

    for tag in soup(["script", "style", "noscript", "nav", "footer", "header"]):
        tag.decompose()

    article = soup.select_one("[class*='ArticleBody'], article")
    content = article or soup
    paragraphs = content.find_all("p")
    text = " ".join(
        paragraph.get_text(" ", strip=True)
        for paragraph in paragraphs
        if paragraph.get_text(strip=True)
    )

    if not text:
        text = content.get_text(" ", strip=True)

    # extract and clean body
    cleaner = getattr(cleaning, "clean_text", None)
    body = cleaner(text) if callable(cleaner) else " ".join(text.split())

    return author, title, body, published_at


# Process the HTML and save the extracted values to df
extracted = df["html"].map(extract_and_clean)
extracted = pd.DataFrame(
    extracted.tolist(),
    index=df.index,
    columns=["extracted_author", "extracted_title", "extracted_body", "extracted_published_at"],
)

df["author"] = extracted["extracted_author"].combine_first(df['author'])
df["title"] = extracted["extracted_title"].combine_first(df["title"])
df["body"] = extracted["extracted_body"].combine_first(df["body"])
df["published_at"] = extracted["extracted_published_at"].combine_first(
    pd.to_datetime(df["published_at"], errors="coerce")
)

In [7]:
print(df.loc[3])

url                    https://news.google.com/rss/articles/CBMijAFBV...
title                  Op-ed: Will China's President Xi’s big bet pay...
published_at                                   2021-09-19 12:00:04+00:00
author                                                   Frederick Kempe
body                   Chinese President Xi Jinping is making the mos...
status                                                           success
error                                                                NaN
discovery_title        Op-ed: Will China's President Xi’s big bet pay...
discovery_published                        Sun, 19 Sep 2021 07:00:00 GMT
discovery_query                                site:cnbc.com geopolitics
html                   <!DOCTYPE html><html lang="en" prefix="og=http...
scrape_http_status                                                 200.0
scrape_error                                                        None
Name: 3, dtype: object


In [ ]:
from filtering.text_filtering import filter_news
filtered_df = filter_news(df)

filtered_df.loc[0]

{'title': [], 'body': ['conflict', 'summit'], 'discovery_title': []}